# AutoNR on Colab: one checked CPU run

Upload this notebook to Colab.

1. Select a **CPU** runtime. High RAM is optional. A GPU does not accelerate this implementation.
2. Run cells in order. Upload the adjacent **autonr_colab_inputs.zip** when asked. No Drive access or repository credentials are needed.
3. Setup checks source hashes, your allocated CPU and a small native AutoNR test before the real fit.
4. The fit uses the raw full input, seed 0 and native search settings.
5. Keep the notebook open while running. Colab can terminate a runtime. There are no artificial fit time or memory limits in this notebook.
6. Run the final download cell **even if setup or fitting fails**. Keep the downloaded results ZIP and its printed SHA256.

This is a portable Linux check, not a claim of equal runtime or bitwise-identical results. The notebook installs a separate environment with pinned CPU dependencies; these differ from the local environment and are recorded. Do not bypass a failed source/test check. This notebook calls the supplied standalone rather than reimplementing AutoNR.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, platform, subprocess, sys, time, zipfile
from google.colab import files

STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
BASE = Path('/content') / ('autonr_' + STAMP)
BASE.mkdir(exist_ok=False)
EVIDENCE = BASE / 'evidence'
EVIDENCE.mkdir()
WORK = BASE / 'project'
WORK.mkdir()
CHECKED = False
RUN_PROCESS = None
print('Session folder:', BASE)
print('If a later cell fails, skip to the final download cell.')

def sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def logged(command, name, cwd=None, env=None):
    with (EVIDENCE / name).open('w') as stream:
        result = subprocess.run(command, cwd=cwd, env=env, stdout=stream, stderr=subprocess.STDOUT, text=True)
    print((EVIDENCE / name).read_text(errors='replace')[-6000:])
    if result.returncode:
        raise RuntimeError(f'{name} failed; download evidence rather than changing pins.')


## 1. Upload the input bundle
Choose `autonr_colab_inputs.zip` from the same local folder as this notebook. The archive includes the exact runner, shared loader, tests and required input files. Extraction checks the archive hash and every file; nothing is downloaded from a repository.


In [ ]:
EXPECTED_ZIP = 'f694804064a575b7d85057f8835c01f0c4ff43f91192ec4cc3b2e6d454ad91bc'
EXPECTED_FILES = {'src/algo/autonr/autonr_standalone.py': 'b5a0c8d1a5ee20c9fb7677c279c41db101d55ed739df3748918f6d7835bbf129', 'src/data.py': 'c1c74a555e6b125ba3e3f6e774332ada34048bed2d436625a276d77451d8eac5', 'tests/test_autonr_standalone.py': 'b5904d84b4085754af12f78033dd58b25b6e3fd126351f27565cdd6165aff59d', 'data/phosphatase/phosphatase/tmp/prot_t5.h5': '425e005d783d70c5ac9043ad2c7d794ce603a8ddbecffa405d569aeddb78c7c1', 'data/phosphatase/phosphatase/data_standard.parquetbundle': 'b5b70771232a442955ef57ec2bb29a18340cdb79f1f4d968c5f7879117689053', 'data/phosphatase/phosphatase/tmp/sequences.fasta': 'c806be15f5eb32c2fe5fdf89c1174df2e8b0fa2b43873d19a86516c03e71149f'}
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly the AutoNR input ZIP.')
name, payload = next(iter(uploaded.items()))
if hashlib.sha256(payload).hexdigest() != EXPECTED_ZIP:
    raise ValueError('Wrong or changed ZIP. Use the bundle supplied with this notebook.')
archive = BASE / 'inputs.zip'
archive.write_bytes(payload)
with zipfile.ZipFile(archive) as z:
    expected_names = set(EXPECTED_FILES) | {'transfer_manifest.json'}
    if len(z.namelist()) != len(expected_names) or set(z.namelist()) != expected_names:
        raise ValueError('Unexpected, duplicate or missing archive entries')
    for member in z.infolist():
        target = (WORK / member.filename).resolve()
        if not target.is_relative_to(WORK.resolve()):
            raise ValueError('Unsafe archive path')
        if member.filename in EXPECTED_FILES and hashlib.sha256(z.read(member)).hexdigest() != EXPECTED_FILES[member.filename]:
            raise ValueError('Input hash mismatch: ' + member.filename)
    z.extractall(WORK)
(EVIDENCE / 'transfer_manifest.json').write_bytes((WORK / 'transfer_manifest.json').read_bytes())
del uploaded, payload
print('Verified runner, tests and input files.')


## 2. Install an isolated Python environment

Installation may take a few minutes. Python 3.12.10 is provisioned separately, so Colab's notebook packages remain unchanged. ClustPy 0.0.2 is installed without its unused neural dependencies: this CPU path needs no PyTorch. The subsequent import, source and real-fit tests must pass.

The CPU dependency pins are a portable environment choice, not new algorithm settings. If installation fails, download the logs for review; do not relax versions or edit library code.


In [ ]:
logged([sys.executable, '-m', 'pip', 'install', 'uv==0.7.13'], 'install_uv.log')
UV = [sys.executable, '-m', 'uv']
logged(UV + ['python', 'install', '3.12.10'], 'install_python.log')
VENV = BASE / 'venv'
logged(UV + ['venv', '--python', '3.12.10', str(VENV)], 'create_env.log')
PY = str(VENV / 'bin/python')
pins = ['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.6.1',
        'pandas==2.2.3', 'matplotlib==3.10.3', 'h5py==3.13.0',
        'pyarrow==19.0.1', 'psutil==7.0.0', 'threadpoolctl==3.6.0', 'tqdm==4.67.1']
(EVIDENCE / 'requested_packages.json').write_text(json.dumps(pins + ['clustpy==0.0.2'], indent=2))
logged(UV + ['pip', 'install', '--python', PY] + pins, 'install_dependencies.log')
logged(UV + ['pip', 'install', '--python', PY, '--no-deps', 'clustpy==0.0.2'], 'install_clustpy.log')
logged(UV + ['pip', 'freeze', '--python', PY], 'installed_packages.txt')
logged(['git', 'init', '-q'], 'git_init.log', cwd=WORK)
logged(['git', 'add', 'src', 'tests', 'transfer_manifest.json'], 'git_add.log', cwd=WORK)
logged(['git', '-c', 'user.name=AutoNR transfer', '-c', 'user.email=transfer@localhost',
        'commit', '-qm', 'Record supplied AutoNR transfer snapshot'], 'git_commit.log', cwd=WORK)
RUNNER = WORK / 'src/algo/autonr/autonr_standalone.py'
ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['MPLCONFIGDIR'] = str(BASE / 'matplotlib')


## 3. Inspect allocated CPU and choose threading

The hardware shown here is the actual allocation; Pro/high RAM does not guarantee more cores. Leave `THREADS = None` for the numerical libraries' native settings. You may set an explicit positive count if you want a recorded threading experiment. More threads are not necessarily faster. This setting changes execution resources, not AutoNR's repetitions or search bounds.

The synthetic test runtime in the next cell is a correctness check, not an ETA for the full protein fit.


In [ ]:
THREADS = None
if THREADS is not None:
    if not isinstance(THREADS, int) or THREADS < 1:
        raise ValueError('THREADS must be None or a positive integer')
    for key in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
        ENV[key] = str(THREADS)
logged(['lscpu'], 'cpu.txt')
logged(PY.split() + ['-c', 'import json,psutil,os; print(json.dumps(dict(logical_cpus=psutil.cpu_count(),physical_cpus=psutil.cpu_count(logical=False),affinity=psutil.Process().cpu_affinity(),ram_bytes=psutil.virtual_memory().total),indent=2))'], 'hardware.json', env=ENV)
(EVIDENCE / 'thread_choice.json').write_text(json.dumps({'threads': THREADS, 'basis': 'native library settings' if THREADS is None else 'user-selected execution setting'}, indent=2))


## 4. Check source, input and real-library correctness
This performs no full protein fit. A failed check blocks the next cell. The tests include a small AutoNR fit, direct-library agreement, archive validation and cancellation of a disposable test process.


In [ ]:
CHECKED = False
logged([PY, str(RUNNER), 'preflight'], 'preflight.log', cwd=WORK, env=ENV)
logged([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_autonr_standalone.py', '-v'], 'tests.log', cwd=WORK, env=ENV)
CHECKED = True
print('Checks passed. Inspect CPU allocation above before committing to the full fit.')


## 5. Run one full raw-input fit

Run this cell when ready. It starts one fresh seed-0 run, with 15 native internal repetitions, all records, all 1,024 coordinates, MDL outliers enabled and native search bounds. No extra outer seeds or data reduction are added.

Progress prints every 30 seconds, including the latest candidate and resource sample. A candidate can take a long time; repeated log text does not prove a hang. The notebook makes no completion-time promise.

To stop **this Colab attempt**, press the cell's stop button. The handler requests cancellation and cleans up the owned child process tree. Keep the logs and use the download cell afterward. Colab losing the whole VM can lose its files, so download interim evidence if you pause work. There is no mid-search resume checkpoint.


In [ ]:
if not CHECKED:
    raise RuntimeError('Run and pass setup checks first.')
RUN_ID = 'colab_raw_full_seed0_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
ATTEMPT = WORK / 'results/autonr/attempts' / RUN_ID
(EVIDENCE / 'run_selection.json').write_text(json.dumps({'run_id': RUN_ID, 'seed': 0, 'records': 'full', 'role': 'matching environment check'}, indent=2))
with (EVIDENCE / 'driver.log').open('w') as stream:
    RUN_PROCESS = subprocess.Popen([PY, '-u', str(RUNNER), 'run', '--run-id', RUN_ID,
                                    '--seed', '0', '--records', 'full'], cwd=WORK,
                                    env=ENV, stdout=stream, stderr=subprocess.STDOUT)
    try:
        while RUN_PROCESS.poll() is None:
            time.sleep(30)
            for filename in ['stdout.log', 'resources.csv']:
                p = ATTEMPT / filename
                if p.exists():
                    print(filename + ':', '\n'.join(p.read_text(errors='replace').splitlines()[-2:]), flush=True)
        RUN_RETURN_CODE = RUN_PROCESS.wait()
        print('Process exit:', RUN_RETURN_CODE)
        print((EVIDENCE / 'driver.log').read_text(errors='replace')[-3000:])
    except KeyboardInterrupt:
        ATTEMPT.mkdir(parents=True, exist_ok=True)
        (ATTEMPT / 'CANCEL').touch(exist_ok=True)
        print('Cancellation requested for this Colab attempt only.')
        try:
            RUN_PROCESS.wait(timeout=15)
        except subprocess.TimeoutExpired:
            subprocess.run([PY, '-c', 'from src.algo.autonr.autonr_standalone import stop_tree; stop_tree(' + str(RUN_PROCESS.pid) + ')'], cwd=WORK, env=ENV, check=True)
            RUN_PROCESS.wait()
        (EVIDENCE / 'notebook_cancelled.json').write_text(json.dumps({'cancelled': True, 'returncode': RUN_PROCESS.returncode}))
        print('Stopped; run the download cell.')


## 6. Reopen and validate
Run after the fit returns. A missing success marker means it is incomplete, failed or cancelled. Download that evidence too; do not call it a result.


In [ ]:
if 'ATTEMPT' not in globals() or not (ATTEMPT / 'SUCCESS.json').exists():
    print('No validated result yet. Download the evidence below for review.')
else:
    logged([PY, str(RUNNER), 'validate', '--run-id', RUN_ID], 'returned_validation.log', cwd=WORK, env=ENV)
    print((ATTEMPT / 'run.csv').read_text())
    print((ATTEMPT / 'subspaces.csv').read_text())
    print('Full labels, model arrays and search diagnostics are ready to return.')


## 7. Download results or failure evidence

This cell works after failed setup as long as the first cell ran. It includes logs, hardware, installed package versions, source/input hashes, exact source files and any saved result arrays. It excludes the virtual environment and the original input files, which are already available locally.

If a fit is still running, this is an **interim snapshot**, not a final archive. Download again after completion.


In [ ]:
EXPORT = Path('/content') / ('autonr_return_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '.zip')
paths = list(EVIDENCE.rglob('*'))
for folder in ['src', 'tests', 'results']:
    if (WORK / folder).exists():
        paths += list((WORK / folder).rglob('*'))
paths = sorted({p for p in paths if p.is_file() and '__pycache__' not in p.parts})
status = {'snapshot_unix': time.time(), 'interim': RUN_PROCESS is not None and RUN_PROCESS.poll() is None,
          'input_zip_sha256': globals().get('EXPECTED_ZIP'), 'files': {}}
with zipfile.ZipFile(EXPORT, 'x', compression=zipfile.ZIP_DEFLATED) as z:
    for path in paths:
        name = path.relative_to(BASE).as_posix()
        blob = path.read_bytes()
        z.writestr(name, blob)
        status['files'][name] = hashlib.sha256(blob).hexdigest()
    z.writestr('return_manifest.json', json.dumps(status, indent=2))
print('Interim snapshot:', status['interim'])
print('ZIP:', EXPORT.name)
print('SHA256:', sha(EXPORT))
files.download(str(EXPORT))
